In [ ]:
import os
import yaml
import gc
from pathlib import Path

import numpy as np
import pandas as pd

root = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "config.yml").exists())
with open(root / "config.yml") as f:
    DATA_PATH = yaml.safe_load(f)["data_path"]

# 2024

## Variables Needed
* Contextual variables 
* Intervention & Probability
    * Send vs no send: `action_id` = {1: 'Send', 0: 'No Send'}
    * Domain: `message_metric`
    * Chosen message: `core_message_id`
    * Message strategy: `strategy`
    * uniform for send vs no send: `p_send`
    * uniform for strategy & domain of chosen message (without replacement!): `p_mood`, `p_steps`, `p_sleep`
        * these are conditional probabilities P(Domain|Action=1)
* BL Characteristics

### Intervention & Probability

In [ ]:
fnames_probs = {
    '2024b': '2024BehaviorAgentProbabilities.csv',
    # '2025a': '2025AllAgentProbabilities.csv',
    # '2025b': '2025BehaviorAgentProbabilities.csv'
}
probs = {k: pd.read_csv(os.path.join(DATA_PATH, v)) for k, v in fnames_probs.items()}

interventions = probs['2024b'].copy()
p_cols = interventions.columns[interventions.columns.str.startswith('p_')]

# convert to datetime
interventions['recorded_time_utc'] = pd.to_datetime(interventions['recorded_time_utc'], utc=True)
interventions['hour_utc'] = pd.to_datetime(interventions['hour_utc'], utc=True)
interventions['decision_date'] = pd.to_datetime(interventions['decision_date'])

# verify decision_date reflects time in UTC
assert (interventions['decision_date'] == interventions['hour_utc'].dt.date).all()

# verify Preinternship notification => before July 1
p = interventions['notification_id'].str.extract(r"(\D+)\d*(?:-\d+-\d)?$")[0] == 'Preinternship'
q = interventions['recorded_time_utc'] < pd.Timestamp("2024-07-01", tz="UTC")
assert (~p | q).all()
# verify after July 1 => not Preinternship notification
assert (q | ~p).all()

# verify NoMessage notification <=> action_id == 0
p = interventions['notification_id'].str.extract(r"(\D+)\d*(?:-\d+-\d)?$")[0] == 'NoMessage'
q = interventions['action_id'] == 0
assert (~p | q).all()
assert (~q | p).all()

# filter out preinternship
interventions = interventions.loc[interventions['recorded_time_utc'] >= pd.Timestamp("2024-07-01", tz="UTC")]

# verify action_id == 1 <=> core_message_id
assert (~(interventions['action_id'] == 1) | (~interventions['core_message_id'].isna())).all()
assert (interventions['core_message_id'].isna() | (interventions['action_id'] == 1)).all()

# verify probability is not missing
assert interventions[p_cols].notnull().all().all()

# verify there is only 1 notification per day per participant
s = interventions\
    .groupby('participant_id')['decision_date']\
    .apply(lambda s: s.duplicated().sum())
assert (s == 0).all()

# verify no send <=> missing metric & missing strategy
p = interventions['action_id'] == 0
q = interventions['message_metric'].isnull() & interventions['strategy'].isnull()
assert (~p | q).all()
assert (~q | p).all()

# verify chosen message has consistent domain and strategy
assert interventions\
    .groupby('core_message_id')[['message_metric', 'strategy']]\
    .nunique().eq(1).all(axis=1).all()

# verify probabilities add up to one
assert np.isclose(interventions[['p_send', 'p_no_send']].sum(axis=1), 1).all()
assert np.isclose(interventions[['p_mood', 'p_steps', 'p_sleep']].sum(axis=1), 1).all()
assert np.isclose(interventions[
    p_cols.drop(['p_send', 'p_no_send', 'p_mood', 'p_steps', 'p_sleep'])
].sum(axis=1), 1).all()

# verify p_send = p_no_send = 0.5
assert np.isclose(interventions[['p_send', 'p_no_send']], 0.5).all().all()

# subset to relevant columns
interventions = interventions[[
    'decision_id',
    'participant_id',
    'decision_date',
    'hour_utc',
    'action_id',
    'message_metric',
    'strategy',
    *interventions.columns[interventions.columns.str.startswith('p_')]
]]\
.set_index('decision_id')\
.rename(columns={
    'hour_utc': 'decision_time_utc',
    'decision_date': 'decision_date_utc'
})

# export
interventions.to_csv(os.path.join(DATA_PATH, "processed/interventions_2024.csv"))

del fnames_probs, probs, interventions, p, q, s
gc.collect()

### Contextual Variables & Outcomes

In [ ]:
def wide2long(df, regex=r'^(Fitbit|Garmin|Healthkit)', extra=('Mood',), pid='participant_id'):
    """Device columns -> long: one row per index entry and device, one column per stream.

    Keeps all of `df`'s index levels and adds `device_type`. Columns in
    `extra` (e.g. Mood) are not device-specific: they are copied onto every
    device the participant (`pid` index level) has data from, including
    rows where that device recorded nothing.
    """
    keys = list(df.index.names)

    dev = df.filter(regex=regex)
    dev.columns = pd.MultiIndex.from_tuples(
        [c.split('_', 1) for c in dev.columns], names=['device_type', 'outcome']
    )
    long = dev.stack('device_type').dropna(how='all').reset_index()

    extra = [c for c in extra if c in df.columns]
    if extra:
        devices = long[[pid, 'device_type']].drop_duplicates()   # each participant's devices
        ext = df[extra].dropna(how='all').reset_index().merge(devices, on=pid)
        long = long.merge(ext, on=[*keys, 'device_type'], how='outer')

    return long.set_index([*keys, 'device_type']).sort_index()


fnames_dfs = {
    '2024': 'dfHourly2024.csv',
    # '2025': 'dfHourly2025.csv'
}
dfs = {k: pd.read_csv(os.path.join(DATA_PATH, v)) for k, v in fnames_dfs.items()}

contexts = dfs['2024'].copy()
contexts = contexts.rename(columns={
    'time_utc': 'context_time_utc',
    'time_local': 'context_time_local'
})
# get date
contexts['context_time_utc'] = pd.to_datetime(contexts['context_time_utc'])
# TODO: determine what to do with those missing time in UTC
# filter 24 h before the first possible decision
contexts = contexts.loc[contexts["context_time_utc"] >= pd.Timestamp("2024-06-30", tz="UTC")]
# replace 0 mood with NaN
contexts['Mood'] = contexts['Mood'].replace(0, np.nan)

# convert from wide to long
idx = ['participant_id', 'context_time_utc', 'context_time_local']
contexts_wide = contexts.set_index(idx)
contexts_long = wide2long(contexts_wide)
del contexts, contexts_wide
gc.collect()

# combined Healthkit: watch, NaNs filled with phone
watch = contexts_long.xs('HealthkitWatch', level='device_type')
phone = contexts_long.xs('HealthkitPhone', level='device_type')
combined = (
    pd.concat({'Healthkit': watch.combine_first(phone)}, names=['device_type'])
    .reorder_levels([*idx, 'device_type'])
)
contexts_long = pd.concat([contexts_long, combined])

# export
contexts_long.to_csv(os.path.join(DATA_PATH, "processed/contexts_2024.csv"))

del fnames_dfs, dfs, watch, phone, combined, contexts_long
gc.collect()

In [ ]:
# column suffix -> aggregation (first match wins; default is mean)
RULES = {
    "steps": "sum",           # total
    "sleep_minutes": "sum",   # total minutes asleep
    "hr_avg": "mean",
    "hr_max": "max",
    "RHR": "median",          # usually a daily value repeated across hours
    "Mood": "mean",
    "_from_phone": "mean",    # share of hours filled from phone
}
 
# columns that also get a within-window SD
SD_SUFFIXES = ("hr_avg", "Mood")
 
 
def _rule(col: str) -> str:
    return next((how for suffix, how in RULES.items() if col.endswith(suffix)), "mean")


def aggregate(
    df: pd.DataFrame,
    keys: list[str] = ("participant_id", "decision_date_utc"),
    min_step_hours: int | None = None,
) -> pd.DataFrame:
    """Aggregate hourly rows to one row per group, one rule per data stream.
 
    Adds `<col>_sd` = SD of hourly values for each hr_avg and Mood column
    (NaN if fewer than 2 values), and `<col>_n` = number of non-null hours
    per group for each stream. Groups with no data for a stream are NaN
    (not 0, which plain sum gives). If `min_step_hours` is set, step totals
    from groups with fewer hours of step data are set to NaN, since sums
    from partial windows are undercounts.
    """
    keys = list(keys)
    g = df.groupby(keys)
    n = g.count()
    cols = [c for c in df.columns if c not in keys]  # keys may be columns or index levels
    out = g.agg({c: _rule(c) for c in cols}).where(n > 0)
 
    if min_step_hours is not None:
        steps = [c for c in out.columns if c.endswith("steps")]
        out[steps] = out[steps].where(n[steps] >= min_step_hours)
 
    # within-window variability
    sd_cols = [c for c in cols if c.endswith(SD_SUFFIXES)]
    sd = g[sd_cols].std().add_suffix("_sd")
 
    counts = n[[c for c in n.columns if not c.endswith("_from_phone")]]
    return out.join(sd).join(counts.add_suffix("_n"))
 
 
def hour_utc(s: pd.Series) -> pd.Series:
    """Parse to UTC timestamps and floor to the start of the hour."""
    return pd.to_datetime(s, utc=True).dt.floor("h")


def build_window_grid(
    interventions: pd.DataFrame,
    period_h: int = 24,
    pid: str = "participant_id",
    dec_time: str = "decision_time_utc",
) -> pd.DataFrame:
    """One row per participant x window index t, whether or not a decision occurred.

    Each participant's first decision defines t = 0. Each later decision advances t
    by its gap from the previous decision in units of `period_h`, rounded and at
    least 1 (a gap of ~1 day -> +1, ~2 days -> +2), so drifting or shifted decision
    times never collide and skipped days get their own index. Window t starts at the
    decision time when
    there is one; otherwise it is interpolated from the nearest decision in steps of
    `period_h`. Window t ends where window t+1 starts (the last ends period_h later),
    so windows are contiguous and every hour belongs to exactly one window.
    t = -1 is the window before the first decision, used only as history.
    """
    intv = (interventions.assign(**{dec_time: hour_utc(interventions[dec_time])})
                         .sort_values([pid, dec_time]))
    step = pd.Timedelta(hours=period_h)
    # index from consecutive gaps: each decision advances t by its rounded gap
    # (at least 1), so drift in decision times never makes two decisions collide
    gap = intv.groupby(pid)[dec_time].diff() / step
    inc = gap.round().clip(lower=1).fillna(0).astype(int)
    intv["t"] = inc.groupby(intv[pid]).cumsum()
    assert not intv.duplicated([pid, "t"]).any(), "duplicate window index"

    tmax = intv.groupby(pid)["t"].max()
    grid = pd.DataFrame({
        pid: np.repeat(tmax.index.values, tmax.values + 2),
        "t": np.concatenate([np.arange(-1, m + 1) for m in tmax.values]),
    }).merge(intv[[pid, "t", dec_time]], on=[pid, "t"], how="left")

    # interpolate start times for windows without a decision
    known_t = grid["t"].where(grid[dec_time].notna())
    g = grid.groupby(pid)
    f_time, f_t = g[dec_time].ffill(), known_t.groupby(grid[pid]).ffill()
    b_time, b_t = g[dec_time].bfill(), known_t.groupby(grid[pid]).bfill()
    grid["start"] = (f_time + (grid["t"] - f_t) * step).fillna(b_time - (b_t - grid["t"]) * step)

    grid["end"] = grid.groupby(pid)["start"].shift(-1).fillna(grid["start"] + step)
    assert (grid["end"] > grid["start"]).all(), "window start times are not increasing"
    grid["span_h"] = (grid["end"] - grid["start"]) / pd.Timedelta("1h")
    grid["has_decision"] = grid[dec_time].notna()
    return grid


def windows_by_index(
    contexts: pd.DataFrame,
    interventions: pd.DataFrame,
    streams: list[str],
    period_h: int = 24,
    by: list[str] = ("participant_id", "device_type"),
    ctx_time: str = "context_time_utc",
    dec_time: str = "decision_time_utc",
) -> pd.DataFrame:
    """Aggregate hourly contexts into the window grid.

    Returns one row per `by` + t for t >= 0, with
      <feature>_post : window t       (proximal outcome when a decision starts window t)
      <feature>_pre  : window t - 1   (history / moderators)
    plus start, end, span_h, has_decision and the intervention columns
    (missing when there is no decision in window t). Rows exist for every window
    of every device a participant has data from, even windows with no data.
    """
    by = list(by)
    pid = by[0]
    ts = "datetime64[ns, UTC]"
    grid = build_window_grid(interventions, period_h, pid, dec_time)
    grid[["start", "end"]] = grid[["start", "end"]].astype(ts)

    ctx = contexts.assign(**{ctx_time: hour_utc(contexts[ctx_time]).astype(ts)})
    bad = ctx[[pid, ctx_time]].isna().any(axis=1)
    if bad.any():
        print(f"dropping {bad.sum()} context rows with missing {pid} or {ctx_time}")
    ctx = ctx.loc[~bad].sort_values(ctx_time)
    # assign each hour to the window with start <= c, then drop hours past its end
    hourly = pd.merge_asof(
        ctx[[*by, ctx_time, *streams]], grid[[pid, "t", "start", "end"]].sort_values("start"),
        left_on=ctx_time, right_on="start", by=pid, direction="backward",
    )
    hourly = hourly.loc[hourly["t"].notna() & (hourly[ctx_time] < hourly["end"])]
    hourly["t"] = hourly["t"].astype(int)

    # TODO: verify interpretation of sleep = 0 & handle potential edge cases
    feats = aggregate(hourly[[*by, "t", *streams]], keys=[*by, "t"]).reset_index()

    # full grid for every device a participant has data from
    devices = ctx[by].drop_duplicates()
    rows = grid.merge(devices, on=pid)
    post = feats.rename(columns={c: f"{c}_post" for c in feats.columns if c not in [*by, "t"]})
    pre = (feats.assign(t=feats["t"] + 1)
                .rename(columns={c: f"{c}_pre" for c in feats.columns if c not in [*by, "t"]}))
    span_pre = grid[[pid, "t", "span_h"]].assign(t=grid["t"] + 1).rename(columns={"span_h": "span_h_pre"})

    out = (rows.rename(columns={"span_h": "span_h_post"})
               .merge(post, on=[*by, "t"], how="left")
               .merge(pre, on=[*by, "t"], how="left")
               .merge(span_pre, on=[pid, "t"], how="left"))
    out = out.loc[out["t"] >= 0]

    intv = interventions.assign(**{dec_time: hour_utc(interventions[dec_time])})
    out = out.merge(intv, on=[pid, dec_time], how="left")
    return out.set_index([*by, "t"]).sort_index()
 

### BL Characteristics

* age: `Age`
* sex: `Sex`
    * include gender?
* race/ethnicity: `Ethnicity_[1-7,9]`
* marital: `Marital`
* depression history
    * has history: `depr0`
    * (relatives with history): `deprRelat`
    * (age at first experience of depression): `deprAge`
* treatment
    * ever on tx: `deprTreat0`
    * (tx type): `specTreat0_*`
* medication
    * current on med: `medication0`
    * (med type): `medication0_*`
    * current on therapy: `therapy0`
* (specialty): `SpecialtyAAMC`?

In [ ]:
interventions = pd.read_csv(os.path.join(DATA_PATH, "processed/interventions_2024.csv"))
contexts_long = pd.read_csv(os.path.join(DATA_PATH, "processed/contexts_2024.csv"))

intv = interventions.reset_index()   # interventions frame (decision_id, participant_id, hour_utc, ...)
streams = [c for c in contexts_long.columns if c not in ('participant_id', 'context_time_utc', 'context_time_local', 'device_type')]

feats = windows_by_index(contexts_long.reset_index(), intv, streams=streams)

# filter out data before 7/1
feats = feats.loc[(feats['start'] >= pd.Timestamp("2024-07-01", tz="UTC"))]

# read
data = pd.read_csv(os.path.join(DATA_PATH, '2024SurveyData.csv'))
# currently on medication
data['medication0'] = data[data.columns[data.columns.str.match(r"^medication\d_None$")]]\
    .all(axis=1).astype(int) * -1 + 1
# remap sex
data['Sex'] = data['Sex'].map({1: 0, 2: 1})
# subset
bl_cols = ['Age', 'Sex', 'depr0', 'medication0', 'PHQtot0', 'GADtot0', 'Neu0_published', 'SLE0']
bl = data.set_index("PARTICIPANTIDENTIFIER")[bl_cols]
# add BL characteristics
feats = feats.reset_index().merge(
    bl.reset_index(), how='left', left_on='participant_id', right_on='PARTICIPANTIDENTIFIER'
).set_index(["participant_id", "device_type"])

# TODO: convert steps into log scale

# TODO: subset feats to necessary variables

# export
feats.to_csv(os.path.join(DATA_PATH, f"processed/data_final_2024.csv"))

del feats, data, bl
gc.collect()